<div class="alert alert-block alert-info">
    
## Imports & Parameters
</div>

In [1]:
## Import
import os
import re
import glob
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
import dask.dataframe as dd
from datetime import datetime, timedelta

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 300)
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [2]:
# Calculate the start and end dates directly
start_date = datetime.today() - timedelta(days=106)
end_date = datetime.today() - timedelta(days=1)

print('Date range : {} to {}'.format(start_date.strftime('%Y%m%d'), end_date.strftime('%Y%m%d')))

# Generate the list of dates with a 10-day interval
date_list = [start_date.strftime('%Y%m%d')]
current_date = start_date

while current_date <= end_date:
    current_date += timedelta(days=10)  # Increment by 10 days
    if current_date <= end_date:
        date_list.append(current_date.strftime('%Y%m%d'))

print(date_list)

Date range : 20250201 to 20250517
['20250201', '20250211', '20250221', '20250303', '20250313', '20250323', '20250402', '20250412', '20250422', '20250502', '20250512']


In [3]:
# Get the current working directory
cwd = os.getcwd()
local_extract = '/Users/E2074/local-datasets/customer/age-prediction/local_extract/'

print(cwd)
print(local_extract)

/Users/E2074/analytics/customer/age-prediction
/Users/E2074/local-datasets/customer/age-prediction/local_extract/


<div class="alert alert-block alert-info">
    
## Datasets
</div>

<div class="alert alert-block alert-warning">

### Metric Data
</div>

In [4]:
def get_metric_data():
    print('Query 1 running..')
    start_time = time.time()
    
    active_customer_base = f"""
    
        WITH funnel_data AS (
            
            SELECT
                customerid,
                SUM(ao_sessions_unique_daily) AS ao_sessions_unique_daily,
                SUM(fe_sessions_unique_daily) AS fe_sessions_unique_daily,
                SUM(rr_sessions_unique_daily) AS rr_sessions_unique_daily,
                SUM(gross_rides_daily) AS gross_rides_daily,
                SUM(net_rides_daily) AS net_rides_daily,
                SUM(active_day) AS active_day
            FROM
                (
                SELECT 
                    day,
                    customerid,
                    MAX(coalesce(gross_rides_daily, ao_sessions_unique_daily)) ao_sessions_unique_daily,
                    MAX(coalesce(gross_rides_daily, fe_sessions_unique_daily)) fe_sessions_unique_daily,
                    MAX(coalesce(gross_rides_daily, rr_sessions_unique_daily)) rr_sessions_unique_daily,
                    SUM(gross_rides_daily) gross_rides_daily,
                    SUM(net_rides_daily) net_rides_daily,
                    MAX(active_day) active_day
                FROM 
                    datasets.customer_rf_daily_kpi 
                WHERE 
                    day >= date_format(current_date - interval '120' day, '%Y-%m-%d')
                    AND day <= date_format(current_date, '%Y-%m-%d')
                    AND customerid NOT IN (SELECT customer_id FROM experiments_internal.customer_predicated_age_immutable WHERE confidence_tag != 'VERY_HIGH')
                    -- AND customerid = '6288c6694281e53bbe33d133' 
                GROUP BY 1,2
                )
            GROUP BY 1
            )
                
            SELECT * FROM funnel_data
            
    """
    
    df_active_customer_base = pd.read_sql(active_customer_base, connection)
    end_time = time.time()
    execution_time = round(((end_time - start_time)*1.00/60.00), 2)
    print("Execution time:", execution_time, "mins")
    df_active_customer_base.head(1)
    df_active_customer_base.to_parquet(local_extract + 'raw_customer_funnel.parquet', index=False)

In [5]:
# get_metric_data()

<div class="alert alert-block alert-warning">

### Appography, Demographic and Segment Data
</div>

In [6]:
def get_customer_demo_data():
    for date in date_list:
        print('Query running for ', date, ' ....')
        start_time = time.time()
        
        customer_demo_data = f"""
            WITH appography AS ( 
            
                SELECT
                    userid,
                    apps_installed
                FROM
                    datasets.customer_apps_installed_snapshot
                WHERE
                    DATE_FORMAT(from_unixtime(updated_epoch / 1000), '%Y%m%d') BETWEEN '{date}' AND
                    DATE_FORMAT(DATE_PARSE('{date}','%Y%m%d') + interval '9' day , '%Y%m%d')
                    -- BETWEEN date_format(current_date - interval '120' day, '%Y%m%d') AND date_format(current_date, '%Y%m%d')
                    AND userid NOT IN (SELECT customer_id FROM experiments_internal.customer_predicated_age_immutable WHERE confidence_tag != 'VERY_HIGH')
                    -- AND userid = '6288c6694281e53bbe33d133' 
                ),
            
                category AS (
                
                SELECT 
                    userid,
                    ai,
                    cameraFilter,
                    courier,
                    dating,
                    deliveryFood,
                    deliveryGrocery,
                    devotional,
                    driverApp,
                    ecommerce,
                    educational,
                    entertainment,
                    fantasySports,
                    financeInvestment,
                    financeNews,
                    financeTransactions,
                    gaming,
                    government,
                    health,
                    homeSecurity,
                    insurance,
                    jobSearch,
                    news,
                    office,
                    ott,
                    social,
                    streamingMusic,
                    telecom,
                    tools,
                    travelBookings,
                    travelMetro,
                    travelRidehailing,
                    vehicle,
                    wearable
                FROM 
                    reports.sql_ingestion_customer_appography_category_view
                WHERE
                    yyyymmdd = '20250511'
                    AND userid NOT IN (SELECT customer_id FROM experiments_internal.customer_predicated_age_immutable WHERE confidence_tag != 'VERY_HIGH')
                ),
                
                demographic_data AS ( 
            
                SELECT
                    customerId customer_id,
                    CASE WHEN age BETWEEN 15 AND 60 THEN age ELSE 0 END age,
                    COALESCE(gender, 'UNKNOWN') gender,
                    preferredTimeBucket
                FROM 
                    datasets.customer_single_view
                WHERE 
                    customerId NOT IN (SELECT customer_id FROM experiments_internal.customer_predicated_age_immutable WHERE confidence_tag != 'VERY_HIGH')
                ),
                
                segment AS (
                
                SELECT
                    customer_id,
                    taxi_ltr_segment ltr_segment,
                    taxi_retention_segments retention_segments,
                    taxi_regularity_segment regularity_segment,
                    taxi_service_affinity service_affinity
                    
                FROM 
                    datasets.iallocator_customer_segments
                WHERE 
                    run_date =  date_format(current_date - interval '2' day, '%Y-%m-%d')
                    AND taxi_recency_segment != 'INACTIVE' 
                    AND customer_id NOT IN (SELECT customer_id FROM experiments_internal.customer_predicated_age_immutable WHERE confidence_tag != 'VERY_HIGH')
                )
                
                SELECT
                    appography.userid,
                    demographic_data.age,
                    demographic_data.gender,
                    demographic_data.preferredTimeBucket,
                    appography.apps_installed,
                    coalesce(ai, 0) ai,
                    coalesce(cameraFilter, 0) cameraFilter,
                    coalesce(courier, 0) courier,
                    coalesce(dating, 0) dating,
                    coalesce(deliveryFood, 0) deliveryFood,
                    coalesce(deliveryGrocery, 0) deliveryGrocery,
                    coalesce(devotional, 0) devotional,
                    coalesce(driverApp, 0) driverApp,
                    coalesce(ecommerce, 0) ecommerce,
                    coalesce(educational, 0) educational,
                    coalesce(entertainment, 0) entertainment,
                    coalesce(fantasySports, 0) fantasySports,
                    coalesce(financeInvestment, 0) financeInvestment,
                    coalesce(financeNews, 0) financeNews,
                    coalesce(financeTransactions, 0) financeTransactions,
                    coalesce(gaming, 0) gaming,
                    coalesce(government, 0) government,
                    coalesce(health, 0) health,
                    coalesce(homeSecurity, 0) homeSecurity,
                    coalesce(insurance, 0) insurance,
                    coalesce(jobSearch, 0) jobSearch,
                    coalesce(news, 0) news,
                    coalesce(office, 0) office,
                    coalesce(ott, 0) ott,
                    coalesce(social, 0) social,
                    coalesce(streamingMusic, 0) streamingMusic,
                    coalesce(telecom, 0) telecom,
                    coalesce(tools, 0) tools,
                    coalesce(travelBookings, 0) travelBookings,
                    coalesce(travelMetro, 0) travelMetro,
                    coalesce(travelRidehailing, 0) travelRidehailing,
                    coalesce(vehicle, 0) vehicle,
                    coalesce(wearable, 0) wearable,
                    coalesce(ltr_segment, 'UNKNOWN') ltr_segment,
                    coalesce(retention_segments, 'UNKNOWN') retention_segments,
                    coalesce(regularity_segment, 'UNKNOWN') regularity_segment,
                    coalesce(service_affinity, 'UNKNOWN') service_affinity
                FROM 
                    appography
                LEFT JOIN
                    category
                    ON appography.userid = category.userid
                LEFT JOIN 
                    demographic_data
                    ON appography.userid = demographic_data.customer_id
                LEFT JOIN 
                    segment
                    ON appography.userid = segment.customer_id
        """
        
        df_customer_demo_data = pd.read_sql(customer_demo_data, connection)
        end_time = time.time()
        execution_time = round(((end_time - start_time)*1.00/60.00), 2)
        print("Execution time:", execution_time, "mins")
        df_customer_demo_data.to_parquet(local_extract + 'demo/amj/{}_customer_demo_data.parquet'.format(date), index=False)

In [37]:
get_customer_demo_data()

Query running for  20250201  ....
Execution time: 1.79 mins
Query running for  20250211  ....
Execution time: 2.33 mins
Query running for  20250221  ....
Execution time: 2.11 mins
Query running for  20250303  ....
Execution time: 2.25 mins
Query running for  20250313  ....
Execution time: 2.31 mins
Query running for  20250323  ....
Execution time: 3.02 mins
Query running for  20250402  ....
Execution time: 3.73 mins
Query running for  20250412  ....
Execution time: 4.71 mins
Query running for  20250422  ....
Execution time: 5.9 mins
Query running for  20250502  ....
Execution time: 11.53 mins
Query running for  20250512  ....
Execution time: 8.74 mins


<div class="alert alert-block alert-warning">

### Read Local Extract & Merge
</div>

#### Local Extract

In [7]:
def get_local_extract():

    start_time = time.time()
    # print('customer funnel')
    # df_active_customer_base = pd.read_parquet(local_extract + 'raw_customer_funnel.parquet')
    # df_active_customer_base.rename(columns={'customerid': 'customer_id'}, inplace=True)
    
    print('customer demo')

    # Pattern to match all .parquet files in the demo folder
    demo_parquet_files = glob.glob(os.path.join(local_extract, 'demo/amj', '*.parquet'))

    # Read and concatenate all the Parquet files into one DataFrame
    df_customer_demo_data = pd.concat(
        (pd.read_parquet(file) for file in demo_parquet_files),
        ignore_index=True
    )
    
    
    df_customer_demo_data.rename(columns={'userid': 'customer_id'}, inplace=True)
    
    end_time = time.time()
    execution_time = round(((end_time - start_time)*1.00/60.00), 2)
    print("Execution time:", execution_time, "mins")
    
    return df_customer_demo_data #, df_active_customer_base

In [8]:
df_customer_demo_data = get_local_extract()

customer demo
Execution time: 1.19 mins


In [9]:
df_customer_demo_data.shape

(9540838, 42)

In [10]:
df_customer_demo_data.customer_id.nunique()
# 1092918

9540838

In [11]:
df_customer_demo_data.head(1)

,customer_id,age,gender,preferredTimeBucket,apps_installed,ai,cameraFilter,courier,dating,deliveryFood,deliveryGrocery,devotional,driverApp,ecommerce,educational,entertainment,fantasySports,financeInvestment,financeNews,financeTransactions,gaming,government,health,homeSecurity,insurance,jobSearch,news,office,ott,social,streamingMusic,telecom,tools,travelBookings,travelMetro,travelRidehailing,vehicle,wearable,ltr_segment,retention_segments,regularity_segment,service_affinity
0,5d70a995a24c96105e399f65,38.0,MALE,Afternoon,"aha,adobe lightroom,amazon prime video,amazon ...",0,1,1,0,1,1,0,0,1,0,0,0,0,0,1,1,0,0,0,0,0,0,1,1,1,1,0,1,0,0,1,0,1,PHH,PLATINUM,WEEKLY,ONLY_CAB


In [16]:
9540838/6

1590139.6666666667

In [17]:
for col in df_customer_demo_data.select_dtypes(include=['int64']).columns:
    df_customer_demo_data[col] = df_customer_demo_data[col].astype('int32')

df_customer_demo_data['chunk'] = df_customer_demo_data.index % 6  # Assign chunk numbers (0-4)
dfs = [group.reset_index(drop=True) for _, group in df_customer_demo_data.groupby('chunk')]

print([len(d) for d in dfs])  # Check row counts

[1590140, 1590140, 1590140, 1590140, 1590139, 1590139]


<div class="alert alert-block alert-info">

## Feature Engineering
</div>

### One-Hot-Encoder

In [18]:
categorical_columns = ['gender', 'ltr_segment', 'retention_segments', 'regularity_segment', 'service_affinity', 'preferredTimeBucket']

In [19]:
def one_hot_encoder(df):
    start_time = time.time()
    
    df_categorical = df[categorical_columns].copy()
    
    for column_name in categorical_columns:
        df_categorical[column_name] = df_categorical[column_name].fillna(column_name + '_UNKNOWN')
    
    # Initialize the OneHotEncoder
    encoder = OneHotEncoder()
    
    # Fit and transform the categorical columns
    encoded_categorical = encoder.fit_transform(df_categorical)
    
    # Convert the encoded result into a DataFrame
    encoded_df = pd.DataFrame(encoded_categorical.toarray(), 
                              columns=encoder.get_feature_names_out(input_features=categorical_columns))
    
    # Now drop the original categorical columns from the original DataFrame
    df = df.drop(columns=categorical_columns)

    # Concatenate the original DataFrame with the encoded DataFrame
    df = pd.concat([df, encoded_df], axis=1)

    end_time = time.time()
    execution_time = round(((end_time - start_time)*1.00/60.00), 2)
    print("Execution time:", execution_time, "mins")
    
    return df

### Vectorized Operation (Explode)

In [20]:
def vectorizer(df):
    start_time = time.time()
    
    df_dummies = df['apps_installed'].str.get_dummies(sep=',')
    
    # Convert all dummies to int32 (saves memory)
    df_dummies = df_dummies.astype('int32')

    # Use pd.concat() instead of df.join() (Faster)
    df_vectorized = pd.concat([df.drop(columns=['apps_installed']), df_dummies], axis=1)

    end_time = time.time()
    execution_time = round(((end_time - start_time)*1.00/60.00), 2)
    print("Execution time:", execution_time, "mins")
    
    return df_vectorized

### Bins definitions

In [21]:
bins = [15, 20, 25, 30, 35, 45, 61]
labels = ['Below-21', '21-25',  '26-30', '31-35', '36-45', 'Above-45']

In [22]:
def one_hot_encoder_to_vectorizer(df):
    print('one_hot_encoder')
    df_one_hot_encoded = one_hot_encoder(df)

    df_one_hot_encoded['age'] = df_one_hot_encoded['age'].replace(0, np.nan)

    print('vectorizer')
    df_vectorized = vectorizer(df_one_hot_encoded)

    df_vectorized['age_group'] = pd.cut(df_vectorized['age'], bins=bins, labels=labels, right=False)

    # Find columns with NaN values
    columns_with_nan = df_vectorized.columns[df_vectorized.isna().any()].tolist()
    
    # Filter DataFrame to get only columns with NaN values
    df_with_nan = df_vectorized[columns_with_nan]
    # print(df_with_nan.isnull().sum())

    return df_vectorized

<div class="alert alert-block alert-info">

## Stratified Sampling & Model
</div>

### Classification

In [23]:
def evaluate_target_vs_rest(df, target_column):
    target_classes = df[target_column].unique().tolist()
    target_classes.sort()
    classifiers = {}
    
    for target_class in target_classes:
        
        # Create a copy of df to avoid modifying the original dataframe
        df_temp = df.copy()
        
        # Split the bins target_class vs Rest 
        df_target = df_temp[df_temp[target_column] == target_class]
        df_rest = df_temp[df_temp[target_column] != target_class]
        
        df_other_sample = df_rest.sample(n=df_target.shape[0], random_state=1)
        df_combined = pd.concat([df_target, df_other_sample], ignore_index=True)
        
        # Create a binary target column where target_class is 1 and rest are 0
        df_combined['target'] = df_combined[target_column].apply(lambda x: 1 if x == target_class else 0)

        # Ensure all column names are strings
        df_combined.columns = df_combined.columns.astype(str)

        # train_test_split
        X_train, X_test, y_train, y_test = train_test_split(df_combined.drop(columns=[target_column, 'target']), 
                                                            df_combined['target'], 
                                                            test_size=0.2, 
                                                            random_state=42)

        # Train Random Forest model
        rf_model = RandomForestClassifier(random_state=42)
        rf_model.fit(X_train, y_train)

        classifiers[target_class] = rf_model

        # Extract feature importances
        feature_importances = rf_model.feature_importances_

        # Get the indices of top features
        top_feature_indices = np.argsort(feature_importances)[::-1][:20] # Adjust 20 as needed
        # Get the names of top features
        top_features = X_train.columns[top_feature_indices]

        # Make predictions
        y_pred = rf_model.predict(X_test)

        # Evaluate model performance
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        # Output results
        print(f"Results for class {target_class} vs Rest")
        # print("Accuracy : ",  round(accuracy * 100.00, 2), "%")
        # print("Precision : ", round(precision * 100.00, 2), "%")
        # print("Recall : ", round(recall * 100.00, 2), "%")
        # print("F1 Score : ", round(f1 * 100.00, 2), "%")
        # print("Top Features:")
        
        # for feature in top_features:
            # print(feature)
        # print("---------------------------------------")
        
    return classifiers

In [24]:
def predict_age_group(df, classifiers):
    predictions = pd.DataFrame(index=df.index)
    
    for target_class, classifier in classifiers.items():
        # Ensure the features used in prediction match the training data
        X = df[[col for col in df.columns if col != 'age_group']]
        predictions[f'{target_class}'] = classifier.predict_proba(X)[:, 1]
        
    binary_predictions = (predictions > 0.5).astype(int)
    
    return binary_predictions

### Final Customer Age Class

In [25]:
# Define a function to apply the logic
def age_weight_function(row):
    cols_with_one = [col for col in ['Below-21', '21-25', '26-30', '31-35', '36-45', 'Above-45'] if row[col] == 1]
    total_sum = row['total_sum']
    
    if total_sum == 0:
        return 'UNKNOWN'
    elif total_sum == 1:
        return cols_with_one[0]
    elif total_sum == 2:
        return cols_with_one[-1]
    elif total_sum == 3 or total_sum == 5:
        return cols_with_one[len(cols_with_one) // 2]
    elif total_sum == 4:
        return cols_with_one[2]
    elif total_sum == 6:
        return cols_with_one[4]
    else:
        return 'UNKNOWN'

def get_confidence_tag(row):
    total_sum = row['total_sum']
    
    if total_sum == 0:
        return 'UNKNOWN'
    elif total_sum == 1:
        return 'VERY_HIGH'
    elif total_sum == 2:
        return 'HIGH'
    elif total_sum == 3:
        return 'MEDIUM'
    elif total_sum in [4, 5, 6]:
        return 'LOW'
    else:
        return 'UNKNOWN'

### Model 

In [26]:
def classifiers_n_predictions(df_vectorized, month, i):

    
    df_vectorized_age_w = df_vectorized[~df_vectorized['age_group'].isna()]
    df_vectorized_age_na = df_vectorized[df_vectorized['age_group'].isna()]
    
    # Keep a copy of the columns to be retained
    df_vectorized_age_na_columns = df_vectorized_age_na[['customer_id', 'age']].copy()
    
    # Drop unnecessary columns for model training
    df_vectorized_age_w.drop(columns=['customer_id', 'age'], axis=1, inplace=True)
    df_vectorized_age_na.drop(columns=['customer_id', 'age'], axis=1, inplace=True)

    
    classifiers = evaluate_target_vs_rest(df_vectorized_age_w, 'age_group')
    classifiers

    # Use the models to predict the age group for df_vectorized_age_na
    predictions = predict_age_group(df_vectorized_age_na, classifiers)
    # display(predictions.head(2))

    predictions['total_sum'] = predictions.sum(axis=1)

    # Merge the original columns with the prediction results
    df_vectorized_age_na_with_predictions = pd.concat([df_vectorized_age_na_columns['customer_id'], predictions], axis=1)
    # display(df_vectorized_age_na_with_predictions.head(2))

    print(df_vectorized_age_na_with_predictions.total_sum.value_counts(normalize=True))


    df_vectorized_age_na_with_predictions['age_group'] = df_vectorized_age_na_with_predictions.apply(age_weight_function, axis=1)
    df_vectorized_age_na_with_predictions['confidence_tag'] = df_vectorized_age_na_with_predictions.apply(get_confidence_tag, axis=1)

    
    # print(df_vectorized_age_na_with_predictions.head(2))
    print(df_vectorized_age_na_with_predictions.age_group.value_counts(normalize=True))
    print(df_vectorized_age_na_with_predictions.confidence_tag.value_counts(normalize=True))

    df_final1 = df_vectorized[~df_vectorized['age_group'].isna()][['customer_id', 'age_group']]
    df_final1['confidence_tag'] = 'VERY_HIGH'

    df_final2 = df_vectorized_age_na_with_predictions[['customer_id', 'age_group', 'confidence_tag']]

    df_predicated_age_bins = pd.concat([df_final1,df_final2])
    print(df_predicated_age_bins.shape)

    print(df_predicated_age_bins.confidence_tag.value_counts(normalize=True))

    df_predicated_age_bins['run_date'] = '2025-05-17'
    
    df_predicated_age_bins.to_csv( local_extract + 'final/customer_predicted_age_{}_{}.csv'.format(month,i), index=False)

In [28]:
for i in range(0,6):
    start_time = time.time()
    print(i)
    df_vectorized = one_hot_encoder_to_vectorizer(dfs[i])
    classifiers_n_predictions(df_vectorized, 'apr_may', i)
    end_time = time.time()
    execution_time = round(((end_time - start_time)*1.00/60.00), 2)
    print("Execution time:", execution_time, "mins")

0
one_hot_encoder
Execution time: 0.22 mins
vectorizer
Execution time: 18.77 mins
Results for class 21-25 vs Rest
Results for class 26-30 vs Rest
Results for class 31-35 vs Rest
Results for class 36-45 vs Rest
Results for class Above-45 vs Rest
Results for class Below-21 vs Rest
total_sum
2    0.449920
3    0.366067
1    0.089010
4    0.080543
0    0.008046
5    0.006138
6    0.000276
Name: proportion, dtype: float64
age_group
21-25       0.338687
36-45       0.283354
26-30       0.139301
31-35       0.122287
Above-45    0.088295
Below-21    0.020031
UNKNOWN     0.008046
Name: proportion, dtype: float64
confidence_tag
HIGH         0.449920
MEDIUM       0.366067
VERY_HIGH    0.089010
LOW          0.086958
UNKNOWN      0.008046
Name: proportion, dtype: float64
(1590140, 3)
confidence_tag
VERY_HIGH    0.510664
HIGH         0.241673
MEDIUM       0.196632
LOW          0.046709
UNKNOWN      0.004322
Name: proportion, dtype: float64
Execution time: 42.83 mins
1
one_hot_encoder
Execution time: